# 🧬 DERMADISC-Net — Training Notebook
Fine-tunes an ImageNet-pretrained **EfficientNet-B0** CNN on the **HAM10000** skin-lesion dataset
(≈13.4k dermatoscopic images, 7 classes) and exports it for the DERMADISC Streamlit app.

**How to run (Google Colab, free):**
1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`  (≈15–20 min)
3. Two files download at the end: `dermadisc_model.onnx` and `dermadisc_model_meta.json`.
   Put both in the same folder as `app.py`.

In [ ]:
%pip install -q datasets onnx onnxscript onnxruntime

In [ ]:
import os, sys, json, time, random
from datetime import datetime
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, accuracy_score, f1_score

DRY_RUN = os.environ.get("DERMADISC_DRY_RUN") == "1"   # quick pipeline test with fake data (leave off)
EPOCHS = 1 if DRY_RUN else 10
BATCH = 16 if DRY_RUN else 64
IMG_SIZE = 224
LR = 3e-4
SEED = 42
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "| torch", torch.__version__)
if device == "cpu" and not DRY_RUN:
    print("⚠️ No GPU found - in Colab use Runtime → Change runtime type → T4 GPU (CPU training is very slow).")

## 1. Load the HAM10000 dataset

In [ ]:
RAW_CLASSES = ["actinic_keratoses", "basal_cell_carcinoma", "benign_keratosis-like_lesions",
               "dermatofibroma", "melanocytic_Nevi", "melanoma", "vascular_lesions"]

if DRY_RUN:
    def fake_split(n):
        return [{"image": Image.fromarray(np.random.randint(0, 255, (300, 300, 3), dtype=np.uint8)),
                 "dx": RAW_CLASSES[i % 7]} for i in range(n)]
    splits = {"train": fake_split(70), "validation": fake_split(28), "test": fake_split(28)}
else:
    from datasets import load_dataset
    ds = load_dataset("marmal88/skin_cancer")
    splits = {k: ds[k] for k in ["train", "validation", "test"]}

for k, v in splits.items():
    print(f"{k:>10}: {len(v)} images")

## 2. Class names, clinical info, and pre-processing cache

In [ ]:
def class_info(raw):
    r = raw.lower()
    if "actinic" in r:   return {"name": "Actinic Keratosis", "short": "akiec", "urgency": "medium",
                                 "desc": "Rough, scaly sun-damage patch; a pre-cancerous lesion."}
    if "basal" in r:     return {"name": "Basal Cell Carcinoma", "short": "bcc", "urgency": "high",
                                 "desc": "Most common skin cancer; pearly bump or non-healing sore."}
    if "benign" in r or "keratosis" in r:
                         return {"name": "Benign Keratosis", "short": "bkl", "urgency": "low",
                                 "desc": "Harmless growth such as seborrheic keratosis or solar lentigo."}
    if "dermatofibroma" in r:
                         return {"name": "Dermatofibroma", "short": "df", "urgency": "low",
                                 "desc": "Small, firm, benign skin nodule."}
    if "nev" in r:       return {"name": "Melanocytic Nevus (Mole)", "short": "nv", "urgency": "low",
                                 "desc": "Common benign mole."}
    if "melanoma" in r:  return {"name": "Melanoma", "short": "mel", "urgency": "high",
                                 "desc": "Serious skin cancer - needs urgent dermatologist review."}
    if "vascular" in r:  return {"name": "Vascular Lesion", "short": "vasc", "urgency": "low",
                                 "desc": "Blood-vessel lesion such as angioma or haemorrhage."}
    return {"name": raw.replace("_", " ").title(), "short": raw[:5], "urgency": "medium", "desc": ""}

raw_classes = sorted(set(splits["train"]["dx"] if not DRY_RUN else [x["dx"] for x in splits["train"]]))
class_to_idx = {c: i for i, c in enumerate(raw_classes)}
infos = [class_info(c) for c in raw_classes]
print("Classes:", [i["name"] for i in infos])


def build_cache(split, name):
    """Decode + resize every image once (to 256x256) so training epochs are fast."""
    imgs, labels = [], []
    t0 = time.time()
    for i, ex in enumerate(split):
        imgs.append(ex["image"].convert("RGB").resize((256, 256)))   # PIL default = bicubic
        labels.append(class_to_idx[ex["dx"]])
        if (i + 1) % 2000 == 0:
            print(f"  {name}: {i + 1}/{len(split)} cached ({time.time() - t0:.0f}s)")
    return imgs, labels

cache = {k: build_cache(v, k) for k, v in splits.items()}
counts = Counter(cache["train"][1])
print("Train class counts:", {infos[k]["name"]: counts[k] for k in sorted(counts)})

## 3. Datasets, augmentation and class balancing

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.02),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class CachedDS(Dataset):
    def __init__(self, imgs, labels, tf):
        self.imgs, self.labels, self.tf = imgs, labels, tf
    def __len__(self):
        return len(self.imgs)
    def __getitem__(self, i):
        return self.tf(self.imgs[i]), self.labels[i]

workers = 0 if DRY_RUN else 2
train_dl = DataLoader(CachedDS(*cache["train"], train_tf), batch_size=BATCH, shuffle=True,
                      num_workers=workers, pin_memory=(device == "cuda"), drop_last=True)
val_dl = DataLoader(CachedDS(*cache["validation"], eval_tf), batch_size=BATCH, num_workers=workers)
test_dl = DataLoader(CachedDS(*cache["test"], eval_tf), batch_size=BATCH, num_workers=workers)

# HAM10000 is very imbalanced (~67% moles) -> weight the loss toward rare classes
n_total, k = sum(counts.values()), len(raw_classes)
class_weights = torch.tensor([(n_total / (k * counts.get(i, 1))) ** 0.5 for i in range(k)], dtype=torch.float32)
print("Loss weights:", [round(w, 2) for w in class_weights.tolist()])

## 4. Model: EfficientNet-B0 (transfer learning)

In [ ]:
weights = None if DRY_RUN else models.EfficientNet_B0_Weights.IMAGENET1K_V1
model = models.efficientnet_b0(weights=weights)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, k)
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, epochs=EPOCHS,
                                                steps_per_epoch=max(len(train_dl), 1), pct_start=0.2)
use_amp = device == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## 5. Train

In [ ]:
def evaluate(dl):
    model.eval()
    preds, ys, loss_sum = [], [], 0.0
    with torch.no_grad():
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type="cuda", enabled=use_amp):
                out = model(x)
                loss_sum += criterion(out, y).item() * len(y)
            preds += out.argmax(1).cpu().tolist()
            ys += y.cpu().tolist()
    return loss_sum / max(len(ys), 1), ys, preds

history, best_score, best_state = [], -1, None
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0, run_loss, correct, seen = time.time(), 0.0, 0, 0
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", enabled=use_amp):
            out = model(x)
            loss = criterion(out, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        run_loss += loss.item() * len(y)
        correct += (out.argmax(1) == y).sum().item()
        seen += len(y)

    val_loss, vy, vp = evaluate(val_dl)
    row = {
        "epoch": epoch,
        "train_loss": round(run_loss / max(seen, 1), 4),
        "train_acc": round(correct / max(seen, 1), 4),
        "val_loss": round(val_loss, 4),
        "val_acc": round(accuracy_score(vy, vp), 4),
        "val_balanced_acc": round(balanced_accuracy_score(vy, vp), 4),
    }
    history.append(row)
    print(f"Epoch {epoch}/{EPOCHS} | train loss {row['train_loss']:.3f} acc {row['train_acc']:.3f} | "
          f"val acc {row['val_acc']:.3f} bal-acc {row['val_balanced_acc']:.3f} | {time.time() - t0:.0f}s")

    score = row["val_balanced_acc"] + row["val_acc"]       # balance overall accuracy and rare-class recall
    if score > best_score:
        best_score = score
        best_state = {k_: v.detach().cpu().clone() for k_, v in model.state_dict().items()}
        print("   ✓ new best model")

model.load_state_dict(best_state)

## 6. Test-set evaluation

In [ ]:
_, ty, tp = evaluate(test_dl)
names = [i["name"] for i in infos]
labels_idx = list(range(k))
report = classification_report(ty, tp, labels=labels_idx, target_names=names, output_dict=True, zero_division=0)
cm = confusion_matrix(ty, tp, labels=labels_idx)
metrics = {
    "test_accuracy": round(accuracy_score(ty, tp), 4),
    "test_balanced_accuracy": round(balanced_accuracy_score(ty, tp), 4),
    "test_macro_f1": round(f1_score(ty, tp, average="macro", zero_division=0), 4),
    "test_weighted_f1": round(f1_score(ty, tp, average="weighted", zero_division=0), 4),
}
print(json.dumps(metrics, indent=2))
print(classification_report(ty, tp, labels=labels_idx, target_names=names, zero_division=0))

## 7. Export to ONNX (small, fast, no PyTorch needed in the app)

In [ ]:
model.eval().cpu()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
export_kwargs = dict(input_names=["input"], output_names=["logits"],
                     dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17)
try:
    torch.onnx.export(model, dummy, "dermadisc_model.onnx", dynamo=False, **export_kwargs)
except TypeError:   # older torch without the `dynamo` argument
    torch.onnx.export(model, dummy, "dermadisc_model.onnx", **export_kwargs)

# Verify the ONNX model gives the same output as PyTorch
import onnxruntime as ort
sess = ort.InferenceSession("dermadisc_model.onnx", providers=["CPUExecutionProvider"])
onnx_out = sess.run(None, {"input": dummy.numpy()})[0]
with torch.no_grad():
    torch_out = model(dummy).numpy()
print("ONNX vs PyTorch max difference:", float(np.abs(onnx_out - torch_out).max()))
print(f"Model size: {os.path.getsize('dermadisc_model.onnx') / 1e6:.1f} MB")

meta = {
    "model_name": "DERMADISC-Net",
    "architecture": "EfficientNet-B0 (ImageNet pretrained, fine-tuned)",
    "framework": f"PyTorch {torch.__version__} -> ONNX",
    "dataset": "HAM10000 (marmal88/skin_cancer on Hugging Face)",
    "dry_run": DRY_RUN,
    "trained_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "device": device,
    "epochs": EPOCHS, "batch_size": BATCH, "learning_rate": LR, "img_size": IMG_SIZE,
    "mean": MEAN, "std": STD,
    "preprocess": "resize to 256x256 (bicubic) -> resize to 224x224 (bilinear) -> normalize",
    "augmentation": "random resized crop, h/v flips, rotation 20deg, colour jitter",
    "loss": "class-weighted cross-entropy + label smoothing 0.1",
    "optimizer": "AdamW + OneCycle LR",
    "split_sizes": {kk: len(v[0]) for kk, v in cache.items()},
    "train_class_counts": {infos[i]["name"]: counts.get(i, 0) for i in range(k)},
    "raw_classes": raw_classes,
    "classes": names,
    "class_info": infos,
    "metrics": metrics,
    "history": history,
    "per_class": {n: {m: round(report[n][m], 4) for m in ["precision", "recall", "f1-score"]} | {"support": int(report[n]["support"])}
                  for n in names},
    "confusion_matrix": cm.tolist(),
    "model_size_mb": round(os.path.getsize("dermadisc_model.onnx") / 1e6, 1),
}
with open("dermadisc_model_meta.json", "w") as f:
    json.dump(meta, f, indent=2)
print("Saved dermadisc_model.onnx and dermadisc_model_meta.json")

## 8. Download the files → put both next to `app.py`

In [ ]:
if "google.colab" in sys.modules:
    from google.colab import files
    files.download("dermadisc_model.onnx")
    files.download("dermadisc_model_meta.json")